In [1]:
# Import Data manipulation libraries and visualization libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Imprt filter warnings to ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Import OrderedDict
from collections import OrderedDict

# Import Loggings 
import logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    filename = 'Phissing.log',
                    filemode = 'w',
                    force = True)

# Import Sklearn libraries for machine learning
from sklearn.model_selection import train_test_split,StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix

# Importing models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost
from xgboost import XGBClassifier

# Setting the Theme style so that it shows no truncations
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
logging.info("Libraries importing Completed")

In [3]:
# Data Ingestion (Importing the dataset)

df = pd.read_csv(r'C:\Phissing_Prediction_Model\data\raw\Phissing_Dataset.csv')
df.head()

,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,nb_eq,nb_underscore,nb_tilde,nb_percent,nb_slash,nb_star,nb_colon,nb_comma,nb_semicolumn,nb_dollar,nb_space,nb_www,nb_com,nb_dslash,http_in_path,https_token,ratio_digits_url,ratio_digits_host,punycode,port,tld_in_path,tld_in_subdomain,abnormal_subdomain,nb_subdomains,prefix_suffix,random_domain,shortening_service,path_extension,nb_redirection,nb_external_redirection,length_words_raw,char_repeat,shortest_words_raw,shortest_word_host,shortest_word_path,longest_words_raw,longest_word_host,longest_word_path,avg_words_raw,avg_word_host,avg_word_path,phish_hints,domain_in_brand,brand_in_subdomain,brand_in_path,suspecious_tld,statistical_report,nb_hyperlinks,ratio_intHyperlinks,ratio_extHyperlinks,ratio_nullHyperlinks,nb_extCSS,ratio_intRedirection,ratio_extRedirection,ratio_intErrors,ratio_extErrors,login_form,external_favicon,links_in_tags,submit_email,ratio_intMedia,ratio_extMedia,sfh,iframe,popup_window,safe_anchor,onmouseover,right_clic,empty_title,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,0,0,0,0,3,0,1,0,0,0,0,1,0,0,0,1,0.000000,0.0,0,0,0,0,0,3,0,0,0,0,0,0,4,4,3,3,3,11,11,6,5.750000,7.0,4.500000,0,0,0,0,0,0,17,0.529412,0.470588,0,0,0,0.875000,0,0.500000,0,0,80.000000,0,100.000000,0.000000,0,0,0,0.0,0,0,0,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,0,0,0,0,5,0,1,0,0,0,0,0,0,0,0,1,0.220779,0.0,0,0,0,0,0,1,0,0,0,0,1,0,4,4,2,19,2,32,19,32,15.750000,19.0,14.666667,0,0,0,0,0,0,30,0.966667,0.033333,0,0,0,0.000000,0,0.000000,0,0,100.000000,0,80.000000,20.000000,0,0,0,100.0,0,0,0,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,3,2,0,0,5,0,1,0,0,0,0,0,1,0,0,0,0.150794,0.0,0,0,0,1,0,3,1,0,0,0,1,0,12,2,2,3,2,17,13,17,8.250000,8.4,8.142857,0,0,0,0,0,0,4,1.000000,0.000000,0,0,0,0.000000,0,0.000000,0,0,100.000000,0,0.000000,0.000000,0,0,0,100.0,0,0,0,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,0,0,0,0,2,0,1,0,0,0,0,0,0,0,0,1,0.000000,0.0,0,0,0,0,0,2,0,0,0,0,1,0,1,0,5,5,0,5,5,0,5.000000,5.0,0.000000,0,0,0,0,0,0,149,0.973154,0.026846,0,0,0,0.250000,0,0.250000,0,0,100.000000,0,96.428571,3.571429,0,0,0,62.5,0,0,0,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,0,0,0,0,5,0,1,0,0,0,0,1,0,0,0,1,0.000000,0.0,0,0,0,0,0,2,0,0,0,0,1,0,6,3,3,3,4,11,7,11,6.333333,5.0,7.000000,0,0,0,0,0,0,102,0.470588,0.529412,0,0,0,0.537037,0,0.018519,1,0,76.470588,0,0.000000,100.000000,0,0,0,0.0,0,0,0,0,1,0,224,8175,8725,0,0,6,legitimate


In [4]:
logging.info("Data Ingestion Completed")

In [5]:
# Checking the information of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11430 entries, 0 to 11429
Data columns (total 89 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   url                         11430 non-null  object 
 1   length_url                  11430 non-null  int64  
 2   length_hostname             11430 non-null  int64  
 3   ip                          11430 non-null  int64  
 4   nb_dots                     11430 non-null  int64  
 5   nb_hyphens                  11430 non-null  int64  
 6   nb_at                       11430 non-null  int64  
 7   nb_qm                       11430 non-null  int64  
 8   nb_and                      11430 non-null  int64  
 9   nb_or                       11430 non-null  int64  
 10  nb_eq                       11430 non-null  int64  
 11  nb_underscore               11430 non-null  int64  
 12  nb_tilde                    11430 non-null  int64  
 13  nb_percent                  114

In [6]:
logging.info("Checked all the description of the dataset")

In [7]:
 # Checking the number of missing values in each column
df.isnull().sum()

url                           0
length_url                    0
length_hostname               0
ip                            0
nb_dots                       0
nb_hyphens                    0
nb_at                         0
nb_qm                         0
nb_and                        0
nb_or                         0
nb_eq                         0
nb_underscore                 0
nb_tilde                      0
nb_percent                    0
nb_slash                      0
nb_star                       0
nb_colon                      0
nb_comma                      0
nb_semicolumn                 0
nb_dollar                     0
nb_space                      0
nb_www                        0
nb_com                        0
nb_dslash                     0
http_in_path                  0
https_token                   0
ratio_digits_url              0
ratio_digits_host             0
punycode                      0
port                          0
tld_in_path                   0
tld_in_s

In [8]:
logging.info("No missing value found")

In [9]:
# Shape of the Dataset
df.shape

(11430, 89)

### Descriptive Stats

In [10]:
numerical_col = df.select_dtypes(exclude =  'object').columns
categorical_col = df.select_dtypes(include = 'object').columns

# Numerical Descriptive Stats
numerical_stats = []

Q1 = df[numerical_col].quantile(0.25)
Q3 = df[numerical_col].quantile(0.75)
IQR = Q3 - Q1
LW = Q1 - 1.5*IQR
UW = Q3 + 1.5*IQR 
Outlier_Count = (df[numerical_col] < LW) | (df[numerical_col] > UW)
Outlier_Percentage = (Outlier_Count.sum()/len(df))*100

for i in numerical_col:
    num_stats = OrderedDict({
        'Feature': i,
        'Count': df[i].count(),
        'Mean': df[i].mean(),
        'Median': df[i].median(),
        'Std': df[i].std(),
        'Min': df[i].min(),
        'Max': df[i].max(),
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower Whisker' : LW,
        'Upper Whisker' : UW,
        'Outlier_Count': Outlier_Count,
        'Outlier_Percentage': Outlier_Percentage,
    })
    numerical_stats.append(num_stats)
    numerical_stats_df = pd.DataFrame(numerical_stats)

categorical_stats = []

for i in categorical_col:
    cat_stats = OrderedDict({
        'Feature' : i,
        'Count' : df[i].count(),
        'Unique' : df[i].nunique(),
        'Mode' : df[i].mode(), 
    })
    categorical_stats.append(cat_stats)
    categorical_stats_df = pd.DataFrame(categorical_stats)
print(numerical_stats_df, categorical_stats_df)

                       Feature  Count           Mean       Median  \
0                   length_url  11430      61.126684    47.000000   
1              length_hostname  11430      21.090289    19.000000   
2                           ip  11430       0.150569     0.000000   
3                      nb_dots  11430       2.480752     2.000000   
4                   nb_hyphens  11430       0.997550     0.000000   
5                        nb_at  11430       0.022222     0.000000   
6                        nb_qm  11430       0.141207     0.000000   
7                       nb_and  11430       0.162292     0.000000   
8                        nb_or  11430       0.000000     0.000000   
9                        nb_eq  11430       0.293176     0.000000   
10               nb_underscore  11430       0.322660     0.000000   
11                    nb_tilde  11430       0.006649     0.000000   
12                  nb_percent  11430       0.123097     0.000000   
13                    nb_slash  11

In [11]:
logging.info("Descrptive Stats completed")

In [12]:
if 'url' in df.columns:
    df.drop(columns=['url'], inplace=True)
print("Dataset Shape:", df.shape)

Dataset Shape: (11430, 88)


In [13]:
logging.info("Removing Useless columns")

### Dividing the data into X and y

In [14]:
target = 'status'
le = LabelEncoder()
df[target] = le.fit_transform(df[target])
X = df.drop(columns = [target])
y = df[target]

In [15]:
logging.info("X and y separation Done")

### Using the Train and the test split 

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.3, 
                                                    random_state=42, 
                                                    stratify=y)
# stratify = y is used for balancing the classes for the training and testing datasets

In [17]:
logging.info("Splitting into Training and testing data")

### Removing the columns that have only 1 unique values

In [18]:
constant_cols = [col for col in X_train.columns if X_train[col].nunique() == 1]
print("Constant Columns:", constant_cols)
X_train = X_train.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

Constant Columns: ['nb_or', 'ratio_nullHyperlinks', 'ratio_intRedirection', 'ratio_intErrors', 'submit_email', 'sfh']


In [19]:
logging.info("Constant Columns Removal")

### Removing the column that have IQR = 0 which creates unnecessary noise while making ML model

In [20]:
iqr_zero_cols = []
for col in X_train.select_dtypes(include=['int64', 'float64']).columns:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        iqr_zero_cols.append(col)
print("IQR = 0 columns removed:", iqr_zero_cols)
X_train = X_train.drop(columns=iqr_zero_cols)
X_test = X_test.drop(columns=iqr_zero_cols)

IQR = 0 columns removed: ['ip', 'nb_at', 'nb_qm', 'nb_and', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_star', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_com', 'nb_dslash', 'http_in_path', 'ratio_digits_host', 'punycode', 'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain', 'prefix_suffix', 'random_domain', 'shortening_service', 'path_extension', 'nb_external_redirection', 'phish_hints', 'domain_in_brand', 'brand_in_subdomain', 'brand_in_path', 'suspecious_tld', 'statistical_report', 'login_form', 'iframe', 'popup_window', 'onmouseover', 'right_clic', 'empty_title', 'domain_in_title', 'whois_registered_domain', 'dns_record']


In [21]:
logging.info("IQR = 0 columns removed becuase of no use in ML model")

In [22]:
import pickle

features = X_train.columns.tolist()

with open("features.pkl", "wb") as f:
    pickle.dump(features, f)

print("Features saved successfully!")
print("Total features used:", len(features))

Features saved successfully!
Total features used: 38


In [23]:
logging.info("Features Pickle file saved")

### Using the pipeline for Imputing the numerical and categorical columns

In [24]:
numerical_cols = X_train.select_dtypes(exclude='object').columns
categorical_cols = X_train.select_dtypes(include='object').columns

# 2. Create preprocessing pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Create preprocessor
preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

# 4. Create pipeline with SMOTE then PCA
final_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('pca', PCA(n_components=0.95, random_state=42))
])

# 5. For TRAINING data: Apply preprocessing, SMOTE, then PCA
print("Processing training data...")
# First apply preprocessing
X_train_preprocessed = final_pipeline.named_steps['preprocessor'].fit_transform(X_train)

# Then apply SMOTE (this creates synthetic samples)
X_train_smoted, y_train_resampled = final_pipeline.named_steps['smote'].fit_resample(
    X_train_preprocessed, y_train
)
print(f"Original training size: {len(X_train)} → After SMOTE: {len(X_train_smoted)}")

# Finally apply PCA
pca = final_pipeline.named_steps['pca']
X_train_resampled = pca.fit_transform(X_train_smoted)
print(f"After PCA: {X_train_resampled.shape[1]} components")

# 6. For TEST data: Apply preprocessing and PCA only (NO SMOTE!)
print("\nProcessing test data...")
X_test_preprocessed = final_pipeline.named_steps['preprocessor'].transform(X_test)
X_test_transformed = pca.transform(X_test_preprocessed)  # Use same PCA fitted on training


Processing training data...
Original training size: 8001 → After SMOTE: 8002
After PCA: 15 components

Processing test data...


In [25]:
logging.info("Preprocessing Done")

In [26]:
# After applying PCA
print(f"Original number of features: {X_train_smoted.shape[1]}")
print(f"Number of components after PCA: {X_train_resampled.shape[1]}")
print(f"Reduced from {X_train_smoted.shape[1]} to {X_train_resampled.shape[1]} features")

Original number of features: 38
Number of components after PCA: 15
Reduced from 38 to 15 features


In [27]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=1),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=200, random_state=1),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=200, random_state=1),
    "XGBoost": XGBClassifier(n_estimators=200, random_state=1, eval_metric='logloss'),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=1),
    "Bagging": BaggingClassifier(n_estimators=200, random_state=1)
}

results = []

for name, model in models.items():

    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_transformed)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })
results_df = pd.DataFrame(results).sort_values(by='F1 Score', ascending=False)
print(results_df)

                Model  Accuracy  Precision    Recall  F1 Score
4             XGBoost  0.947215   0.945899  0.948658  0.947276
0        RandomForest  0.938758   0.940797  0.936406  0.938596
1          ExtraTrees  0.938174   0.942807  0.932905  0.937830
6             Bagging  0.936716   0.939003  0.934072  0.936531
3    GradientBoosting  0.931467   0.931195  0.931739  0.931467
2            AdaBoost  0.905220   0.900750  0.910735  0.905715
5  LogisticRegression  0.898513   0.889840  0.909568  0.899596


In [28]:
logging.info("Model Trained")

### Top 3 Best Models are found to be 
### 1. XGBoost
### 2. Random Forest
### 3. Extra Tree Classifier
### So applying the K-Fold technique to cross validate them

In [29]:
best_models = {
    "XGBoost": XGBClassifier(n_estimators=200, random_state=1, eval_metric='logloss'),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=1),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=200, random_state=1)
}

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

cv_results = []

for name, model in best_models.items():

    scores = cross_val_score(
        model,
        X_train_resampled,
        y_train_resampled,
        cv=kfold,
        scoring='f1'
    )

    cv_results.append({
        "Model": name,
        "Mean F1 Score": scores.mean(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(by="Mean F1 Score", ascending=False)

print(cv_results_df)

          Model  Mean F1 Score
1  RandomForest       0.940546
0       XGBoost       0.940108
2    ExtraTrees       0.938341


In [30]:
logging.info("K- Fold Applied")

### Applying Hyperparameter Tuning

In [32]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score

rf_params = {
    'n_estimators': [100,200,300,400],
    'max_depth': [None,10,20,30],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'max_features': ['sqrt','log2']
}

xgb_params = {
    'n_estimators': [100,200,300,400],
    'max_depth': [3,4,5,6],
    'learning_rate': [0.01,0.05,0.1],
    'subsample': [0.7,0.8,1],
    'colsample_bytree': [0.7,0.8,1],
    'gamma': [0,0.1,0.2]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=1),
    param_distributions=rf_params,
    n_iter=25,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=1
)

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=1, eval_metric='logloss'),
    param_distributions=xgb_params,
    n_iter=25,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=1
)

rf_search.fit(X_train_resampled, y_train_resampled)
xgb_search.fit(X_train_resampled, y_train_resampled)

best_rf = rf_search.best_estimator_
best_xgb = xgb_search.best_estimator_

print("Best RF Params:", rf_search.best_params_)
print("Best XGB Params:", xgb_search.best_params_)

rf_pred = best_rf.predict(X_test_transformed)
xgb_pred = best_xgb.predict(X_test_transformed)

rf_f1 = f1_score(y_test, rf_pred)
xgb_f1 = f1_score(y_test, xgb_pred)

print("RandomForest F1 Score:", rf_f1)
print("XGBoost F1 Score:", xgb_f1)

if rf_f1 > xgb_f1:
    final_model = best_rf
    print("Final Model: RandomForest")
else:
    final_model = best_xgb
    print("Final Model: XGBoost")

Best RF Params: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}
Best XGB Params: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.8}
RandomForest F1 Score: 0.9405563689604686
XGBoost F1 Score: 0.9468147282291058
Final Model: XGBoost


In [33]:
logging.info("Hyperparameter Tuning applied for checking of underfitting or overfitting")

In [34]:
feature_names = final_pipeline.named_steps['preprocessor'].get_feature_names_out()

print("Features after preprocessing:")
print(feature_names)

print("Total:", len(feature_names))

Features after preprocessing:
['num__length_url' 'num__length_hostname' 'num__nb_dots' 'num__nb_hyphens'
 'num__nb_slash' 'num__nb_www' 'num__https_token' 'num__ratio_digits_url'
 'num__nb_subdomains' 'num__nb_redirection' 'num__length_words_raw'
 'num__char_repeat' 'num__shortest_words_raw' 'num__shortest_word_host'
 'num__shortest_word_path' 'num__longest_words_raw'
 'num__longest_word_host' 'num__longest_word_path' 'num__avg_words_raw'
 'num__avg_word_host' 'num__avg_word_path' 'num__nb_hyperlinks'
 'num__ratio_intHyperlinks' 'num__ratio_extHyperlinks' 'num__nb_extCSS'
 'num__ratio_extRedirection' 'num__ratio_extErrors'
 'num__external_favicon' 'num__links_in_tags' 'num__ratio_intMedia'
 'num__ratio_extMedia' 'num__safe_anchor' 'num__domain_with_copyright'
 'num__domain_registration_length' 'num__domain_age' 'num__web_traffic'
 'num__google_index' 'num__page_rank']
Total: 38


In [35]:
print("Original features:", df.shape[1])
print("Features after cleaning:", X_train.shape[1])
print("Features after preprocessing:", len(feature_names))
print("PCA components:", X_train_resampled.shape[1])

Original features: 88
Features after cleaning: 38
Features after preprocessing: 38
PCA components: 15


In [36]:
sample = X_test_transformed[199].reshape(1, -1)

In [37]:
prediction = final_model.predict(sample)
print("Prediction:", prediction)

Prediction: [1]


In [38]:
if prediction[0] == 0:
    print("Legitimate Website")
else:
    print("Phishing Website")

Phishing Website


In [39]:
print("Actual:", y_test.iloc[100])

Actual: 1


In [40]:
import pickle

pickle.dump(preprocessor, open("preprocessor.pkl", "wb"))
pickle.dump(pca, open("pca.pkl", "wb"))
pickle.dump(final_model, open("final_model.pkl", "wb"))

In [41]:
print(X_train.columns.tolist())

['length_url', 'length_hostname', 'nb_dots', 'nb_hyphens', 'nb_slash', 'nb_www', 'https_token', 'ratio_digits_url', 'nb_subdomains', 'nb_redirection', 'length_words_raw', 'char_repeat', 'shortest_words_raw', 'shortest_word_host', 'shortest_word_path', 'longest_words_raw', 'longest_word_host', 'longest_word_path', 'avg_words_raw', 'avg_word_host', 'avg_word_path', 'nb_hyperlinks', 'ratio_intHyperlinks', 'ratio_extHyperlinks', 'nb_extCSS', 'ratio_extRedirection', 'ratio_extErrors', 'external_favicon', 'links_in_tags', 'ratio_intMedia', 'ratio_extMedia', 'safe_anchor', 'domain_with_copyright', 'domain_registration_length', 'domain_age', 'web_traffic', 'google_index', 'page_rank']


In [42]:
import pandas as pd

df = pd.DataFrame(columns=features)

df.to_csv("test_input.csv", index=False)

In [43]:
test_data = [[
37,19,3,0,3,1,1,0,3,0,4,4,3,3,3,11,11,6,5.75,7,4.5,17,0.5,0.4,0,0.8,0.5,0,80,100,0,0,1,45,-1,0,1,4
]]

df = pd.DataFrame(test_data, columns=features)

df.to_csv("test_input.csv", index=False)